# Diffuison 的神经网络

前五章的内容，实际上都在讲解生成图式模型的基本逻辑。我们默认了一个神经网络可以拟合我们想要的内容，无论是矢量场还是分数场。现在我们来讨论这个神经网络本身。

我为你介绍 Latent Diffusion Model 框架，但是在此之前需要重新介绍 VAE。原因是，LDM 认为直接在原始高维像素空间上操作非常困难，于是提出利用 VAE 建立高维像素空间与低维语义流形潜空间之间的关系，使得仅仅研究低维潜空间成为可能，这大大减少了计算量。在第二章中我们已经简单介绍过 VAE 的一些内容，我们在此基础上继续。

# 重读 Variational Autoencoder

这里给出原论文地址 https://arxiv.org/abs/1312.6114 Auto-Encoding Variational Bayes 作者讲述了 VAE 的训练与推理。在第二章中，我们指出 VAE 构建了从潜空间到真实像素空间之间的映射。潜空间 $\mathcal{Z}$ 是一个低维, 抽象, 规则的数学空间（通常是 $\mathbb{R}^d$），每一个坐标点 $z$ 代表真实空间中的某种图片的语义信息。数据空间 $\mathcal{X}$ 则是高维，具体，杂乱的像素空间（$\mathbb{R}^D$，$D \gg d$），每一个坐标点 $x$ 则代表了真实空间的具体图片。

为了构建这种映射，我们给出 VAE 的具体结构。我们需要训练两个神经网络。对于潜空间内隐变量 $z$，定义编码模型 $q_\phi(z|x)$，接收真实数据 $x$ 给出在潜空间内对应的隐变量概率分布，对应 Encoder。类似的，定义生成模型 $p_\theta(x|z)$，接收潜空间内隐变量 $z$ 给出对应的真实空间内数据概率分布情况，对应 Decoder。Encoder 的参数 $\phi$ 将与 Decoder 的参数 $\theta$ 一起进行联合训练。

需要注意的是，Encoder 和 Decoder 并不直接输出上述的概率分布场。Encoder 接收真实数据点 $x$，输出隐变量分布的均值 $\mu$ 和方差 $\sigma$ 或 $\log \sigma^2$，以此描述隐变量的高斯分布。关于隐变量分布是高斯分布，这其实是一个假设，我们会在训练中尽量维护 VAE 的 Encoder 将真实数据映射到高斯分布。Decoder 则一般接收潜空间数据点 $z$ 直接输出对应的真实数据点 $\hat{x}$，部分方案的 Decoder 会类似 Encoder 输出真实变量分布的均值和方差。

回顾第二章中的推导结果 $$\ln p_\theta(x) = \text{ELBO} + D_{KL}(q_\phi(z|x) \parallel p_\theta(z|x))$$
其中 ELBO 是 $$\text{ELBO} = \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \ln \frac{p_\theta(x, z)}{q_\phi(z|x)} \right]$$
KL 离散度则是 $$D_{KL}(q_\phi(z|x) \parallel p_\theta(z|x)) = \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \ln \frac{q_\phi(z|x)}{p_\theta(z|x)} \right]$$

如果根据 Bayes 公式 $p_\theta(x, z) = p_\theta(x|z)p_\theta(z)$ 展开 ELBO，会得到 $$\text{ELBO} = \underbrace{\mathbb{E}_{z \sim q_\phi(\mathbf{z}|\mathbf{x})} [\ln p_\theta(\mathbf{x}|\mathbf{z})]}_{\text{Reconstruction Loss}} - \underbrace{D_{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p_\theta(\mathbf{z}))}_{\text{Regularization (KL)}}$$

正式定义损失函数 $$\mathcal{L}(\theta,\phi ; x) = -ELBO$$
在明确损失函数的前提下，问题就是实际训练中 ELBO 究竟如何采样？我们下面给出办法。

### 训练算法

假设输入一个 Batch 的数据 $\mathbf{X}^M = \{\mathbf{x}^{(i)}\}_{i=1}^M$。输入 $\mathbf{x}^{(i)}$ 到 Encoder，Encoder 输出分布参数均值 $\boldsymbol{\mu}^{(i)}$ 和方差的对数 $\log(\boldsymbol{\sigma}^{(i)})^2$。

从标准高斯分布中采样噪声 $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$，从而构造潜变量 $\mathbf{z}^{(i)} = \boldsymbol{\mu}^{(i)} + \boldsymbol{\sigma}^{(i)} \odot \boldsymbol{\epsilon}$，其中 $\odot$ 是 Hadamard 积。

将采样得到的 $\mathbf{z}^{(i)}$ 输入 Decoder，Decoder 输出重构结果 $\hat{\mathbf{x}}^{(i)}$，即生成模型对 $\mathbf{x}^{(i)}$ 的预测。

计算损失。此时损失是 $$\mathcal{L}(\theta, \phi ;x)^{(i)} = D_{KL}(q_\phi(\mathbf{z}|\mathbf{x}^{(i)}) \parallel p_\theta(\mathbf{z})) - \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x}^{(i)})} [\log p_\theta(\mathbf{x}^{(i)}|\mathbf{z})]$$

对于 KL 离散度项，由于概率分布是高斯分布，我们直接给出解析解 $$\text{Loss}_{KL} = -\frac{1}{2} \sum_{j=1}^J \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$其中 $J$ 是隐空间的维度。

对于重构项，同样由于高斯分布，写出 $$\text{Loss}_{Recon} = \frac{1}{2\sigma_{obs}^2} \| \mathbf{x}^{(i)} - \hat{\mathbf{x}}^{(i)} \|^2 + C$$
其中 $\frac{1}{2\sigma_{obs}^2}$ 是一个固定的超参数，原因是假设 $p_\theta(x|z)$ 服从高斯分布 $\mathcal{N}(\hat{x}, \sigma_{obs}^2\mathbf{I})$，一般直接设置 $\frac{1}{2\sigma_{obs}^2} = 1$。如果我们采取了 Decoder 也输出真实数据空间中高斯分布均值与方差的方案，那么此处 $\sigma_{obs}$ 就是 Decoder 输出的方差。

所以最终损失是两者之和。对于 Batch 中每个元素累加该损失取平均，然后反向传播更新梯度。

我为你推导上面两个公式。我们先明确潜空间先验分布 $p(z) = \mathcal{N}(0, \mathbf{I})$，神经网络给出的后验分布具备形式 $q_\phi(z|x) = \mathcal{N}(\mu, \sigma^2)$。

由于$$D_{KL}(q \parallel p) = \int q(z) \log \frac{q(z)}{p(z)} dz = \int q(z) \left( \log q(z) - \log p(z) \right) dz$$
而 $\log q(z) = -\frac{1}{2} \log(2\pi\sigma^2) - \frac{(z-\mu)^2}{2\sigma^2}$ 且 $\log p(z) = -\frac{1}{2} \log(2\pi) - \frac{z^2}{2}$。

积分得到 $$D_{KL} = \int q(z) \left[ -\frac{1}{2} \log \sigma^2 - \frac{(z-\mu)^2}{2\sigma^2} + \frac{z^2}{2} \right] dz$$
利用期望的性质 $\mathbb{E}[z^2] = \mu^2 + \sigma^2$ 且 $\mathbb{E}[(z-\mu)^2] = \sigma^2$
$$D_{KL} = -\frac{1}{2} \log \sigma^2 - \frac{1}{2} + \frac{1}{2}(\mu^2 + \sigma^2) = \frac{1}{2} (\mu^2 + \sigma^2 - \log \sigma^2 - 1)$$
最终 $$\text{Loss}_{KL} = -\frac{1}{2} \sum_{j=1}^J \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$

对于重构项，直接展开即证 $$\log p_\theta(x|z) = \log \left[ \frac{1}{\sqrt{2\pi\sigma_{obs}^2}^D} \exp \left( -\frac{\|x - \hat{x}\|^2}{2\sigma_{obs}^2} \right) \right]$$ $$\log p_\theta(x|z) = -\frac{D}{2} \log(2\pi\sigma_{obs}^2) - \frac{1}{2\sigma_{obs}^2} \|x - \hat{x}\|^2$$
不过既然原式是期望，为什么最终结果没有 Monte Carlo 采样？实际上是有的，但是仅仅采样了一次，原因是原作者发现在 Batch 足够大时损失会自动走向无偏真实结果，而无需在每个数据样本单独采样多次。更多的，如果需要采样更多次，实际上只需要采样更多次噪声 $\epsilon$ 计算新的 $z$，将新的 $z$ 输入 Decoder，再根据结果重新计算重构项。需要注意，由于 KL 离散度项仅仅与 Encoder 输出的均值方差有关，无需重新计算。

Encoder 和 Decoder 联合训练其实颇有难度，很容易崩溃。一个著名的现象是后验塌陷 (Posterior Collapse)。如果 Decoder 是一个极其强大的自回归模型（比如强大的 PixelCNN），不需要 $z$ 的信息，只靠自己就能生成合理的图像，那么就会导致 Encoder 的训练低效。另一件事是如果 KL 散度项的权重太大，Encoder 会为了降低 KL 损失而放弃编码选择逼近原始高斯分布，直接输出 $q_\phi(z|x) = \mathcal{N}(0, \mathbf{I})$。

关于潜空间的维度 $J$，这是我们人为设定的超参数。实际上需要根据任务复杂程度决定该参数大小。对于简单的任务，如 MNIST 图片编码语义流形，$J$ 可以设置得极低；对于复杂的编码任务则要调高这一参数。

我们来说推理。

### 推理

VAE 的推理正是 LDM 所看重的。VAE 作为一个生成模型时，Encoder 会被直接弃用，仅使用 Decoder 部分。从预设的先验分布 $p_\theta(z)$（通常是正态分布 $\mathcal{N}(0, \mathbf{I})$）中随机抽取一个潜变量 $z$，输入 Decoder，Decoder 输出生成数据的分布参数或直接输出重构值 $\hat{x}$。作为生成模型，VAE 可以从无意义的噪声中产生符合原始数据分布的新样本。

不过我们更看重的是作为编码模型的能力，此时被弃用的则是 Decoder。真实数据 $x$ 输入 Encoder，Encoder 输出该数据在潜空间的分布参数，即均值 $\mu$ 和方差 $\sigma$。在推理场景下，为了获得确定的特征表达，通常直接取均值 $\mu$ 作为该数据点的潜表征。我们获得了真实数据样本在低维潜空间中的对应。

还有一种用途是数据重构，比如压缩图片。简而言之就是把上面的步骤结合了 $$x \xrightarrow{\text{Encoder } q_\phi} (\mu, \sigma) \xrightarrow{\text{Sampling } z} z \xrightarrow{\text{Decoder } p_\theta} \hat{x}$$

我们接近说完了。所以 VAE 到底用来做什么？在 VAE 将高维真实图片编码为低维潜空间上数据之后，Diffusion 模型可以只学习低维潜空间之间的变换。以 Flow Matching 为例，我们原本让 FM 学习矢量场 $v: [0, 1] \times \mathbb{R}^D \to \mathbb{R}^D$，计算量非常大。然而在使用 VAE 建立真实空间与潜空间之间映射之后，就可以让 FM 只学习矢量场 $v': [0, 1] \times \mathbb{R}^d \to \mathbb{R}^d$，其中 $D \gg d$。在 FM 完成潜空间上推理之后，Decoder 再将推理结果映射回真实空间，这是 Latent Diffusion Model 的基本思想。

# Latent Diffusion Model

推荐你读 https://arxiv.org/abs/2112.10752 High-Resolution Image Synthesis with Latent Diffusion Models 比起之前的各种论文，神经网络本体架构的论文可读性会高很多，我想重要原因是数学推导更少了。

原文基本就是把我们刚刚说的思想展开叙述。这里引出一个概念 Two-Stage Image Synthesis，指的就是通过 VAE 编码器和我们训练的潜空间上扩散模型完成对原始图片编码的过程。在 Perceptual Compression Stage，VAE 的 Encoder 将真实数据空间中图片编码到低维潜空间，而在 Semantic Compression Stage，交由我们训练的扩散模型将上一步压缩的潜表征转移到预期的潜空间位置。其实就是详细描述了我们上一节所说的过程。

我们详谈一些细节。

LDM 指出，对于感知压缩阶段，承担主要功能的是 Encoder $\mathcal{E}$ 和 Decoder $\mathcal{D}$，目的是将原始 RGB 图像 $x \in \mathbb{R}^{H \times W \times 3}$ 压缩为潜变量 $z = \mathcal{E}(x) \in \mathbb{R}^{h \times w \times c}$。此处的 Encoder 和 Decoder 直接采用 VAE 方案是可行的，但是作者指出了改进方案。

具体来说，我们重新定于 VAE 的损失函数$$\mathcal{L}_{Autoencoder} = \mathcal{L}_{regular} + \mathcal{L}_{recconstrcut} +   \lambda \mathcal{L}_{adversarial}$$
对比一下我们原本的损失 $$\mathcal{L}(\theta, \phi ;x) = D_{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \parallel p_\theta(\mathbf{z})) - \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})} [\log p_\theta(\mathbf{x}|\mathbf{z})]$$
解释一下，第一项正则损失并没有变化，仍是原本的 KL 离散度损失。但是第二项重构损失项不再是 $-\mathbb{E}[\log p_\theta(\mathbf{x}|\mathbf{z})]$ 而是变成了 $$\mathcal{L}_{pixel} = \| \mathbf{x} - \tilde{\mathbf{x}} \|_1$$ 以及 $$\mathcal{L}_{perceptual} = \sum_i \frac{1}{H_i W_i C_i} \| \Phi_i(\mathbf{x}) - \Phi_i(\tilde{\mathbf{x}}) \|_2^2$$
这两项的和。后一项被称为 $LPIPS$ 度量损失。其中 $\Phi$ 来自预训练的特征图生成网络，一般是一个 VGG 网络，该网络提取深层特征。原因是对比两张图片相似程度仅仅关注像素级别是不足够的。

最后一项是对抗损失项，我们引入一个判别器 GAN 网络 $Deter$。$Deter$ 通常是一个卷积网络，接收一张图片，输出一个数值，代表它认为这张图是真实图片的概率。如果是 PatchGAN，则输出一个得分矩阵，分别对应原图每个切片部分得分 (例如将原图分割为 $16 \times 16$，每个小方形打一个得分)，表示该位置真实程度，最后总分是取平均。损失表达式则是$$\mathcal{L}_{adv} = - \mathbb{E} \log Deter(\tilde{\mathbf{x}})$$
需要注意的是，$Deter$ 实际上也参与训练。判别器拥有单独的训练轮次和损失函数 $$\mathcal{L}_{Deter} = - [\mathbb{E} \log Deter(\mathbf{x}) + \mathbb{E} \log (1 - Deter(\tilde{\mathbf{x}}))]$$
所以这里其实是 $Deter$ 和 Decoder 的对抗训练。流程是，对于真实图片 $\mathbf{x}$，输入 Decoder 获得 $\tilde{\mathbf{x}}$，再计算 $\mathcal{L}_{Deter}$，遍历一个 Batch 之后取损失平均反向传播更新参数，在此期间将其他网络都视为常数。之后再根据上述的损失函数，将 $Deter$ 视为常数训练 Encoder 和 Decoder，同样遍历一个 Batch 之后反向传播更新参数。

还有一个细节是，LDM 的作者建议 Encoder 使用全卷积架构，换句话说，Encoder 不使用任何线性层。这样做的好处是保持了原始图像的高维特征而不是展平为一维向量。

如你所见，VAE 的训练实际上是 $3$ 个网络的联合训练，即 $\mathcal{E},\mathcal{D},{Deter}$，因此难度极高。在构建 LDM 模型时，我们一般不会自己训练 VAE 网络，而是直接采取工业标准成熟的预训练 VAE。

下面是一张来自 LDM 原论文的图片，详细展示了 LDM 的训练流程。首先最左侧是真实数据所在像素空间，通过 VAE Encoder 与 Decoder 来对高维像素空间和低维潜空间之间进行映射。在真实数据 $x$ 映射到潜空间中 $z$ 之后，经过前向加噪变为时间步 $T$ 下状态 $z_T$。右侧则是有条件生成的处理区域，我们训练一个编码器 $\tau_\theta$，针对不同的输入模态，如文本使用 CLIP，图像使用 CNN 提取器等等，将其转换为中间特征向量，跟随潜表征 $z_T$ 一起被神经网络处理。中间绿色部分粗略展示了 Diffusion 网络，此处使用的是 U-Net，执行 DDIM 的去噪推理过程。最终得到的潜表征会经 Decoder 重新映射回真实像素空间。

<img src="./assets/LDM.png" width="700" height="330">

你可能注意到下方的 Switch，这是为了保证模型既可以处理有条件生成也可以处理无条件生成。我们的训练顺序是先训练 VAE 的部分，再训练 U-Net 学习在潜空间中去噪，同时训练 $\tau_\theta$ 学习如何将条件对齐。一般来说，我们会直接采取现成预训练的 VAE Encoder 与 Decoder，这样可以将精力完全投入中间的 Diffusion 模型训练工作。

对于条件向量融入 U-Net 的处理，作者建议使用 Cross-Attention。后续我们会详谈中间 Diffusion 模型的部分，包括本章的主要内容 Diffuison Transformer。

关于潜空间维数的选择，定义 VAE 下采样率 $f = H/h = W/w$，最终的实验结论是 $f=4$ 到 $f=8$ 具有巨大优势，效率和质量都可以保证。也就是说潜空间维数推荐为原像素空间的 $1/16 \sim 1/64$，根据最后一维通道数量还会波动。

我们已经讲完了 LDM 所有重要内容。现在我们来探讨中间的 Diffusion Model 本身。

# Diffusion Transformer

在 DiT 被发掘之前，工业界广泛采取的内核神经网络方案是 U-Net。U-Net 得名于其结构很像一个字母 $U$。下面一张图详细展示了这一结构。

<img src="./assets/UNet.png" width="600" height="800">

简而言之，上方是时间步 $t$ 和有条件生成的条件 $c$ 嵌入处理区域，左侧则是对于加噪图像的处理区域，最终在右侧输出一个预测的噪声 (DDIM 架构)。

U-Net 架构提供了基本的时间步与条件嵌入图像生成的思路，然而这并不是并不是我们的探讨重点。我们指出 U-Net 的重大缺陷，过度工程化导致难以放大参数量。一个经验规律是，CNN 面对模型参数量放大呈现明显的边际效用，然而 Transformer 遵循符合直觉的 Scaling Law。随着训练计算量和网络参数量的增大，Transformer 的表现可预计地上升，这意味着 Transformer 巨大的可扩展性。

另一件事是，U-Net 本身的 $U$ 型结构非常难以放大。当尝试添加更多层，我们必须重新设计整个连接，包括其中的大量 Concat 操作。

所以 DiT 的进化方向非常了然，我们需要可以平庸地堆叠且主要采用 Transformer 模块的网络结构。

推荐你读 https://arxiv.org/abs/2212.09748 Scalable Diffusion Models with Transformers DiT 原文。下面这张图给出核心的 DiT 架构。

<img src="./assets/DiT.png" width="1000" height="480">

我们分块详细说说这张图所展示的结构。

### 输入与输出

最左下角是初始噪声张量, 时间步与条件的输入。Noised Latent 指的是潜空间中的初始高斯噪声，经过 Patchify 输入层层堆叠的 DiT Block 进行处理，最后输出预测的噪声均值和方差。解释一下什么是 Patchify，简而言之就是将原始 $(H, W, C)$ 张量先沿着 $H \times W$ 平面分割成 $P \times P$ 小块。图像被切成 $L = \frac{H}{P} \times \frac{W}{P}$ 个块，每块展平后的一维向量长度是 $P^2 \times C$，此时张量形状是 $(L, P^2 \cdot C)$。通过一个权重矩阵 $W \in \mathbb{R}^{(P^2 \cdot C) \times D}$ 将每个块映射到模型的隐藏维度 $D$。最终结果的张量维度是 $(L, D)$。

如果我们加上 Batch 维度，实际上是四维张量 $(B, C, H, W)$ 变形为三维张量 $(B, L, D)$。

一个细节是，Patchify 这一步实际上还包含了位置编码的处理，原因是当我们将三维张量展平之后，我们注定需要弥补空间降维损失的信息量。现在的展平导致模型并不清楚这些像素天生的上下左右关系。DiT 原文中指出，这一步使用的是 2D 正余弦位置编码。如果你还不知道什么是 2D 正余弦编码，简而言之就是将正余弦编码的理念扩展到二维。我们简单说说。

假设 Patch 位于网格的第 $i$ 行、第 $j$ 列，模型的 Embedding 维度为 $D$。我们将 $D$ 分成两个 $D/2$ 的子空间。$x$ 方向编码对应列索引 $j$，生成维度为 $D/2$ 的向量 $PE_x(j)$。$y$ 方向编码则对应行索引 $i$，生成维度为 $D/2$ 的向量 $PE_y(i)$。

对于位置 $pos$（即 $i$ 或 $j$）和维度索引 $k \in [0, D/4 - 1]$ $$PE(pos, 2k) = \sin\left(\frac{pos}{10000^{\frac{4k}{D}}}\right)$$ $$PE(pos, 2k+1) = \cos\left(\frac{pos}{10000^{\frac{4k}{D}}}\right)$$

最终该 Patch 的位置编码向量 $PE(i, j) \in \mathbb{R}^D$ 为：$$PE(i, j) = [PE_y(i) \, \Vert \, PE_x(j)]$$其中 $\Vert$ 表示向量拼接 (Concatenation)。

这很容易理解，简而言之就是对于 $x$ 轴与 $y$ 轴做两次正余弦编码再拼接。关于正余弦编码本身的一些特性，我们不多赘述，默认你了解。或者简单一说，正余弦编码是 Transformer 原论文提出的编码方法。利用正弦函数与余弦函数按照如上方式编码时间步或者更多标量的好处是 $PE(pos + \Delta)$ 可以表示为 $PE(pos)$ 的线性变换，而神经网络天然适合学习这种简单的线性变换。

理所应当的，这里的位置编码方式还可以升级为旋转位置编码 (RoPE)。我们同样默认你知道关于旋转位置编码的基础知识，但是我们同样简单一说。

旋转位置编码的核心思想是不再对原始向量做位置编码而是对注意力矩阵 $Q$ 与 $K$ 做位置编码。在一维的旋转位置编码情形下，我们约定旋转角度$$\theta_i = 10000^{-2i/d}, \quad i \in \{0, 1, \dots, d/2 - 1\}$$

注意力矩阵 $Q$ 与 $K$ 的张量形式是 $(L,d)$。对于每一行的向量，我们将其每两个分为一组，按照刚刚约定的角度乘上位置 $m$ 旋转。更详细的，假设注意力矩阵 $Q$ 的第 $m+1$ 行向量形式是 $$q_i = (q_{m,0}, q_{m,1}, \dots, q_{m,d})$$
那么 $(q_{m,2i},q_{2i+1})$ 就需要旋转角度 $m \cdot \theta_i$ 得到新的向量 $(q'_{m,2i},q'_{2i+1})$，最后将全部旋转后向量拼接即可。将整个注意力矩阵 $Q$ 做该操作之后，我们得到了经过未知编码的新注意力矩阵 $Q'$。同理处理注意力矩阵 $K$ 得到 $K'$。之后的注意力部分使用新的注意力矩阵替代原注意力矩阵即可。需要注意的是，由于编码进行在注意力层而不是原向量上，每个 Transformer 模块都需要做位置编码处理。


以上是一维旋转位置编码的处理。此处 Patchify 之后的结果是二维的。实际上二维位置编码完全同理，只是我们将前 $d/2$ 维分配用于 $x$ 轴编码而后 $d/2$ 维分配用于 $y$ 轴编码，然后按照一维编码方式进行即可。我们不赘述这部分内容。

我们现在来说核心的 DiT Block。

### DiT Block

我们先来看原论文最推崇的 DiT Block 设计方案，被称为 adaLN-Zero，也就是最左侧的灰色框内部分。

#### adaLN-Zero

关于每一个组件做了什么，其实一目了然。需要注意的是，每个 DiT Block 内都重复使用了 Embedding 的条件信息，通过 MLP 将其转化成 $6$ 个参数向量，这 $6$ 个参数通过 Scale 和 Shift 的方式作用到经过处理的张量上。这部分被称为 Adaptive Layer Norm。我们详细说说。

在每一个 DiT Block 内部，Scale 和 Shift 作用于 Layer Norm 之后 $$x_{norm} = \text{LayerNorm}(x)$$ 此时特征被映射到均值为 0，方差为 1 的分布。

我们对其进行调制 $$\text{Modulate}(x_{norm}, \gamma, \beta) = x_{norm} \cdot (1 + \gamma) + \beta$$
此处的 $\gamma$ 和 $\beta$ 是由底部的 Conditioning MLP 根据时间步 $t$ 和条件 $y$ 生成的向量。

观察 DiT Block 的架构图，实际上这样的调制一共出现了 $3$ 次。第一次是 $\gamma_1, \beta_1$ 出现在 Attention 模块之前，控制模型在计算自注意力之前的特征分布。第二次 $\gamma_2, \beta_2$ 出现在 Pointwise Feedforward 前，控制模型在进行非线性映射之前的特征分布。第三对 $\alpha_1, \alpha_2$ 出现在残差连接之前，将 Block 的计算结果加回主干之前，对其进行一次总体的缩放。

关于 Conditioning MLP 的权重初始化，这里使用一种特殊的初始化技巧，被称为 Zero-initializing the final linear layer。这也是 adaLN-Zero 得名由来。我们详细说说。

实际上就是直接将 Conditioning MLP 的最后一层权重初始化全部归零，这就保证 $6$ 个参数向量初始时全部为 $0$。这里遵循的思想是，DiT Block 初始化即恒等映射。这使得信号能够在初始时刻毫无阻碍地流过整个网络，极大地降低了深层网络初始训练的难度。随着训练的进行，Conditioning MLP 的权重会更新，各个参数也会逐渐被学习。

更多的，DiT Block 中除了 FFN 之外的所有层中流通的张量形式都是 $(L, D)$，因此这些参数的形式是 $(1, D)$，采用广播的 Hadamard 积与逐点加法和张量做运算。

我们基本说完了。

但是这里有个疑问：如果 $(1, D)$ 不足以表达给出的的时间步 $t$ 和条件 $y$ 中信息怎么办？实际上这种信息瓶颈确实且绝对存在。在生成复杂内容时，$(1, D)$ 的张量大小完全不够用。我们介绍架构图中第二种方案，也就是 Cross-Attention。

#### Cross-Attention

Cross-Attention 在此处的应用其实相当直白。我们将注意力模块中的 $Q$ 保持来自图像 Patch 不变，形状为 $(L, D)$。但是注意力矩阵 $K$ 与 $V$ 改为来自时间步与条件文本编码，形状为 $(L_{label}, D)$。然后做与自注意力机制完全相同的事情，最终输出张量形状是 $(L,D)$。

此处有个细节非常有趣：为什么注意力矩阵 $V$ 来自时间步与条件文本编码而不是图像 Patch？原因首先是矩阵乘法天生的维度，为了使最终产出的张量形状与初始输入图像形状一致为 $(L,D)$，注意力矩阵 $V$ 形状为 $(L_{label}, D)$ 是理所应当的。其次是，此处注意力模块的核心目的是提取文本特征而不是图像已有特征。

最后我们来说说最右边的 DiT Block 方案，也就是 In-Context Conditioning。

#### In-Context Conditioning

最后一种方案采取了极端的简单的方案，直接拼接向量。图像 Patch 序列是形状为 $(L, D)$ 的张量，时间步与条件文本经过编码后形成形状为 $(N, D)$ 张量。我们直接将两者在 $L$ 维度上拼起来，得到一个长度为 $N + L$ 的长序列，形状为 $(N+L, D)$。

在向量拼接之后，张量进入一个标准的自注意力模块。拼接的向量操作允许自注意力自由地注意图像信息与条件信息之间的关联。

需要注意，我认为原论文此处的图实际上画错了。如果采取最后一种拼接向量方案，实际上全程向量拼接只会发生一次，也就是说流动的张量维度一直是 $(N+L, D)$。但是论文中图却很容易误解为每个 DiT Block 都需要拼接一次，这很荒谬，因为残差链接要求了维度尽量不变。

关于 DiT Block，我们基本说完了。一个意外的事实是，虽然原始的 DiT 论文推荐 adaLN-Zero 架构，现在最强大且实用的却是最后一种直接拼接向量的 In-Context Conditioning 方案或者其变体。原因是其图文对齐程度更高且完全统一了架构。至于序列维度增加导致的计算问题，工业界反而研发出诸多方法应对，如 Flash Attention。

# 总结

在 DiT 原文的最后，作者验证了 DiT 上的 Scaling Law，结果完全符合预期。Transformer 架构对于参数量增大天然的性能提升促成了如今强大的图像生成模型。更多的，现在的 State-of-the-Art 模型基本都采取 Flow Matching 作为 Diffusion Solver 而 LDM 加 DiT 作为内核神经网络的方案。

如果你熟悉机器学习中的众多网络模块，如 CNN, MLP 与 Self-Attention，本章的阅读应该会更轻松。

目前为止，我们已经基本交代完毕了 Diffusion 领域的核心 Solver 算法与神经网络架构。未来的篇章中，我们会更侧重讲解一些具体的应用与技术创新。下一章中我们来详谈强大的 Stable Diffusion 3 以及其他实现样例。